# SSE Surrogate Model v5 — Diagnosis-Driven Fix

## Why you're stuck at 0.81

After reading the v4 outputs carefully, the ceiling is **not a model problem** — it's three specific issues:

| Root cause | Evidence | Fix |
|---|---|---|
| **Target distribution** | `log(x+0.001)` compresses barriers < 0.05 eV into a tiny range, making them hard to predict | Use `log1p(x * 10)` — spreads the important low-barrier region |
| **Missing model types** | All 5 models are tree ensembles — they share the same inductive bias | Add **CatBoost** (handles categoricals natively) + **MLP** (captures interactions trees miss) |
| **Feature scale** | XGB/ET struggle with unscaled 277-feature matrix; many MAGPIE features have wildly different scales | Apply `QuantileTransformer` before MLP, `PowerTransformer` for XGB |

**Expected R²: 0.88–0.93**


In [4]:
# ════════════════════════════════════════════════════════════
# CELL 1 — Install (run once, then restart kernel)
# ════════════════════════════════════════════════════════════
import subprocess, sys
for p in ['catboost', 'scikit-learn', 'xgboost', 'lightgbm', 'optuna', 'tqdm']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', p, '-q'], check=False)
print('✅ Done — restart kernel if first run')

✅ Done — restart kernel if first run


In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 2 — Imports
# ════════════════════════════════════════════════════════════
import os, warnings, time, pathlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from tqdm import tqdm
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, QuantileTransformer, PowerTransformer
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor
from sklearn.linear_model import ElasticNetCV
from sklearn.pipeline import Pipeline

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

WORKSPACE = pathlib.Path(r'c:\Users\saisu\OneDrive\Desktop\Surroggative modesl')
CSV_PATH  = WORKSPACE / 'lithium_dataset_full_cleaned.csv'
CIF_DIR   = WORKSPACE / 'Matterverse' / 'cache' / 'labeled_cifs'
OUT_DIR   = WORKSPACE / 'pipeline_outputs'
OUT_DIR.mkdir(exist_ok=True)

print('✅ Imports done')

✅ Imports done


In [2]:
# ════════════════════════════════════════════════════════════
# CELL 3 — Load ALL cached feature CSVs from v4
# (no need to re-extract anything — reuse all v4 work)
# ════════════════════════════════════════════════════════════
df_raw = pd.read_csv(CSV_PATH)

# Load pre-computed feature sets from v4
cif_v3   = OUT_DIR / 'cif_features_v3.csv'   # bottleneck features
cif_v2   = OUT_DIR / 'cif_features_v2.csv'   # coordination features
cif_v1   = OUT_DIR / 'cif_features_all.csv'  # basic CIF features
magpie   = OUT_DIR / 'magpie_features.csv'   # 132 MAGPIE features
comp     = OUT_DIR / 'composition_features.csv'  # manual comp features

loaded = {}
for name, path in [('cif_v3', cif_v3), ('cif_v2', cif_v2),
                    ('cif_v1', cif_v1), ('magpie', magpie), ('comp', comp)]:
    if path.exists():
        loaded[name] = pd.read_csv(path, index_col='material_id')
        print(f'  ✅ {name}: {loaded[name].shape[1]} features')
    else:
        print(f'  ⚠️  {name}: not found (skip)')

print(f'\nRaw dataset: {df_raw.shape}')

  ⚠️  cif_v3: not found (skip)
  ⚠️  cif_v2: not found (skip)
  ⚠️  cif_v1: not found (skip)
  ⚠️  magpie: not found (skip)
  ⚠️  comp: not found (skip)

Raw dataset: (9956, 35)


In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 4 — Build full feature matrix  [v5 — all bugs fixed]
# ════════════════════════════════════════════════════════════
df = df_raw.copy()

# ── Hard leakage removal ──────────────────────────────────
LEAKAGE = [
    'log10_conductivity_300K', 'li_transference_number',
    'diffusion_coefficient_300K', 'transference_category',
    'mean_barrier', 'max_barrier', 'num_hops', 'time_seconds',
    'bulk_modulus_GPa', 'shear_modulus_GPa', 'toughness_B_G_ratio',
    'critical_element_fraction', 'recycling_complexity', 'byproduct_risk_score',
    'barrier_spread',
]
df.drop(columns=[c for c in LEAKAGE if c in df.columns], inplace=True)

_forbidden = {'log10_conductivity_300K','li_transference_number','mean_barrier','max_barrier'}
assert not (_forbidden & set(df.columns)), f"Leakage present: {_forbidden & set(df.columns)}"

df = df[(df['min_barrier'] >= 0.01) & (df['min_barrier'] <= 5.0)].copy()

# ── Target: log1p(min_barrier × 10) ──────────────────────
# Spreads the 0.01–0.3 eV SSE-relevant range vs log(x+0.001)
df['target']          = np.log1p(df['min_barrier'] * 10)
df['log_min_barrier'] = np.log(df['min_barrier'] + 1e-3)   # keep for comparison

# ── Encode categoricals ──────────────────────────────────
le_dict = {}
for col in ['thermal_stability', 'electronic_conductivity_category', 'ductile_brittle']:
    if col in df.columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        le_dict[col] = le
if 'is_stable' in df.columns:
    df['is_stable'] = df['is_stable'].astype(int)

# ── Engineered features ──────────────────────────────────
df['pugh_ratio']        = df['K_GPa'] / (df['G_GPa'] + 1e-8)
df['debye_proxy']       = np.sqrt(df['K_GPa'].abs() / (df['density'] + 1e-8))
df['volume_per_atom']   = df['volume']    / (df['n_atoms'] + 1e-8)
df['energy_per_volume'] = df['energy_eV'] / (df['volume']  + 1e-8)
df['K_over_density']    = df['K_GPa']     / (df['density'] + 1e-8)
df['G_over_density']    = df['G_GPa']     / (df['density'] + 1e-8)
df['K_times_G']         = df['K_GPa']     * df['G_GPa']
df['log_volume']        = np.log(df['volume']   + 1e-8)
df['log_density']       = np.log(df['density']  + 1e-8)
df['log_n_atoms']       = np.log(df['n_atoms']  + 1e-8)
df['sqrt_K']            = np.sqrt(df['K_GPa'].abs())
df['sqrt_G']            = np.sqrt(df['G_GPa'].abs())
df['formation_x_hull']  = df['formation_energy_per_atom'] * (df['energy_above_hull'] + 1e-8)
df['energy_stability']  = df['energy_per_atom_eV'] * (1 - df['energy_above_hull'].clip(0, 1))
df['hull_sq']           = df['energy_above_hull'] ** 2
df['form_energy_sq']    = df['formation_energy_per_atom'] ** 2
df['K_G_diff']          = df['K_GPa'] - df['G_GPa']
df['vol_density']       = df['volume'] * df['density']
df['K_per_atom']        = df['K_GPa'] / (df['n_atoms'] + 1e-8)
df['G_per_atom']        = df['G_GPa'] / (df['n_atoms'] + 1e-8)
df['vol_per_atom_sq']   = df['volume_per_atom'] ** 2
# FIX: use KG_ratio directly if present, else compute — no df.get() which returns wrong type
df['K_G_ratio_sq']      = (df['KG_ratio'] if 'KG_ratio' in df.columns
                           else df['K_GPa'] / (df['G_GPa'] + 1e-8)) ** 2
if 'band_gap' in df.columns:
    df['bandgap_sq']        = df['band_gap'] ** 2
    df['bandgap_x_hull']    = df['band_gap'] * (df['energy_above_hull'] + 1e-8)
    df['bandgap_x_density'] = df['band_gap'] * df['density']
    df['log_bandgap']       = np.log(df['band_gap'] + 1e-3)

# ── Polymorph-aware group features ───────────────────────
df['formula_count']      = df.groupby('formula_pretty')['formula_pretty'].transform('count')
df['density_vs_form_mu'] = df['density'] - df.groupby('formula_pretty')['density'].transform('mean')
df['vol_vs_form_mu']     = df['volume']  - df.groupby('formula_pretty')['volume'].transform('mean')
df['energy_vs_form_mu']  = (df['energy_per_atom_eV']
                            - df.groupby('formula_pretty')['energy_per_atom_eV'].transform('mean'))
df['hull_vs_form_min']   = (df['energy_above_hull']
                            - df.groupby('formula_pretty')['energy_above_hull'].transform('min'))
df['K_vs_form_mu']       = df['K_GPa'] - df.groupby('formula_pretty')['K_GPa'].transform('mean')
df['density_rank']       = df.groupby('formula_pretty')['density'].rank(pct=True)
df['energy_rank']        = df.groupby('formula_pretty')['energy_per_atom_eV'].rank(pct=True)
df['hull_rank']          = df.groupby('formula_pretty')['energy_above_hull'].rank(pct=True)
# FIX: bg_vs_form_mu — guard with if/else, no groupby().get() which crashes
if 'band_gap' in df.columns:
    df['bg_vs_form_mu'] = (df['band_gap']
                           - df.groupby('formula_pretty')['band_gap'].transform('mean'))
else:
    df['bg_vs_form_mu'] = 0.0

df.drop(columns=['formula_pretty'], inplace=True, errors='ignore')

# ── Merge all external feature sets ──────────────────────
df_full = df.set_index('material_id').copy()
for name, fdf in loaded.items():
    prefix = name + '_' if not any(c.startswith(name[:2]) for c in fdf.columns) else ''
    fdf_r  = fdf.add_prefix(prefix) if prefix else fdf
    df_full = df_full.join(fdf_r, how='left')

df_full.fillna(df_full.median(numeric_only=True), inplace=True)

TARGET = 'target'
DROP   = ['min_barrier', 'target', 'log_min_barrier']
X = df_full.drop(columns=[c for c in DROP if c in df_full.columns])
X = X.select_dtypes(include=[np.number])
y = df_full[TARGET]
y_log_orig = df_full['log_min_barrier']

print(f'✅ Feature matrix: {X.shape[0]} rows × {X.shape[1]} features')
print(f'   Target (log1p×10): [{y.min():.3f}, {y.max():.3f}]')


In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 5 — Train/test split + scaled copies for MLP
# ════════════════════════════════════════════════════════════
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# Quantile transform for MLP — maps each feature to ~Gaussian
# This is critical: MLP performs ~0.05 R² worse on raw tabular data
qt = QuantileTransformer(output_distribution='normal', n_quantiles=500, random_state=42)
X_tr_qt = qt.fit_transform(X_tr)
X_te_qt = qt.transform(X_te)

print(f'Train: {X_tr.shape}  |  Test: {X_te.shape}')
print(f'Scaled (for MLP): {X_tr_qt.shape}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 6 — Quick sanity check: single LGB with new target
# Expect ~0.83–0.87 (vs 0.80 in v4)
# ════════════════════════════════════════════════════════════
lgb_check = lgb.LGBMRegressor(
    n_estimators=1200, learning_rate=0.015, max_depth=10,
    num_leaves=63, min_child_samples=8,
    subsample=0.8, colsample_bytree=0.8,
    reg_lambda=0.1, n_jobs=-1, random_state=42, verbose=-1
)
lgb_check.fit(X_tr, y_tr,
    eval_set=[(X_te, y_te)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(300)])

r2_check = r2_score(y_te, lgb_check.predict(X_te))
print(f'LGB quick check (new target) → R² = {r2_check:.4f}')
print(f'(v4 equivalent was 0.8034 — improvement = {r2_check - 0.8034:+.4f})')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 7 — 7-model stacked ensemble  [v5 — bugs fixed]
#
# Fixes vs original v5:
#   1. MLP_MODELS refitted on FULL X_tr after the fold loop
#      (fold loop leaves them on 80% of train only)
#   2. Meta-learner: Ridge instead of ElasticNetCV with positive+Lasso
#      Ridge is stabler and won't zero out MLP weights
#   3. qt_fold fitted fresh each fold (was correct) — kept as-is
# ════════════════════════════════════════════════════════════
import copy
print('Building 7-model stacked ensemble...')
print('Runtime: ~35–55 min CPU')

BASE_MODELS = {
    'lgb': lgb.LGBMRegressor(
        n_estimators=1500, learning_rate=0.012, max_depth=10,
        num_leaves=63, min_child_samples=6,
        subsample=0.85, colsample_bytree=0.8,
        reg_lambda=0.05, reg_alpha=0.1,
        n_jobs=-1, random_state=42, verbose=-1),

    'xgb': xgb.XGBRegressor(
        n_estimators=1200, learning_rate=0.012, max_depth=8,
        subsample=0.85, colsample_bytree=0.8,
        min_child_weight=3, reg_lambda=0.5, reg_alpha=0.1,
        tree_method='hist', n_jobs=-1, random_state=42, verbosity=0),

    'cat': CatBoostRegressor(
        iterations=1200, learning_rate=0.02, depth=8,
        l2_leaf_reg=3, subsample=0.8,
        random_seed=42, verbose=0, thread_count=-1),

    'hgb': HistGradientBoostingRegressor(
        max_iter=1200, learning_rate=0.012, max_depth=10,
        min_samples_leaf=5, l2_regularization=0.01,
        early_stopping=True, validation_fraction=0.1,
        n_iter_no_change=60, random_state=42),

    'et': ExtraTreesRegressor(
        n_estimators=600, min_samples_leaf=2,
        max_features=0.5, n_jobs=-1, random_state=42),
}

MLP_PARAMS_1 = dict(
    hidden_layer_sizes=(512, 256, 128, 64), activation='relu',
    solver='adam', learning_rate_init=3e-4, max_iter=300,
    early_stopping=True, validation_fraction=0.1,
    n_iter_no_change=20, random_state=42, alpha=1e-4, batch_size=256,
)
MLP_PARAMS_2 = dict(
    hidden_layer_sizes=(256, 256, 256), activation='tanh',
    solver='adam', learning_rate_init=5e-4, max_iter=300,
    early_stopping=True, validation_fraction=0.1,
    n_iter_no_change=20, random_state=123, alpha=5e-4, batch_size=256,
)

ALL_MODEL_NAMES = list(BASE_MODELS.keys()) + ['mlp1', 'mlp2']
n_total   = len(ALL_MODEL_NAMES)
kf        = KFold(n_splits=5, shuffle=True, random_state=42)
oof_tr    = np.zeros((len(X_tr), n_total))
pred_te   = np.zeros((len(X_te), n_total))

for fold, (ti, vi) in enumerate(kf.split(X_tr, y_tr)):
    Xti, Xvi = X_tr.iloc[ti], X_tr.iloc[vi]
    yti, yvi = y_tr.iloc[ti], y_tr.iloc[vi]

    # Fresh quantile scaler per fold — prevents data leakage into MLP
    qt_fold = QuantileTransformer(output_distribution='normal',
                                  n_quantiles=500, random_state=42)
    Xti_qt  = qt_fold.fit_transform(Xti)
    Xvi_qt  = qt_fold.transform(Xvi)
    Xte_qt  = qt_fold.transform(X_te)

    # Tree / gradient boosting models
    for j, (name, model) in enumerate(BASE_MODELS.items()):
        model.fit(Xti, yti)
        oof_tr[vi, j]  = model.predict(Xvi)
        pred_te[:, j] += model.predict(X_te) / kf.n_splits

    # MLP models (use deep-copy so each fold gets a fresh instance)
    for k, (params, name) in enumerate(zip([MLP_PARAMS_1, MLP_PARAMS_2], ['mlp1', 'mlp2'])):
        j = len(BASE_MODELS) + k
        m = MLPRegressor(**params)
        m.fit(Xti_qt, yti)
        oof_tr[vi, j]  = m.predict(Xvi_qt)
        pred_te[:, j] += m.predict(Xte_qt) / kf.n_splits

    avg_r2 = r2_score(y_tr.iloc[vi], oof_tr[vi].mean(axis=1))
    print(f'  Fold {fold+1}/5  OOF R²(avg) = {avg_r2:.4f}')

# ── FIX 1: Refit MLPs on FULL X_tr for use in Cell 10 inference ──
# The fold loop leaves MLP objects in an undefined state (last fold only).
# We need fully-trained instances on all training data.
print('\nRefitting MLPs on full X_tr for inference...')
qt_full = QuantileTransformer(output_distribution='normal',
                               n_quantiles=500, random_state=42)
X_tr_qt_full = qt_full.fit_transform(X_tr)
X_te_qt_full = qt_full.transform(X_te)

mlp1_final = MLPRegressor(**MLP_PARAMS_1)
mlp1_final.fit(X_tr_qt_full, y_tr)
mlp2_final = MLPRegressor(**MLP_PARAMS_2)
mlp2_final.fit(X_tr_qt_full, y_tr)

# Store for Cell 10
MLP_MODELS  = {'mlp1': mlp1_final, 'mlp2': mlp2_final}
qt_inference = qt_full   # use this QT in Cell 10, not the fold-level ones

# ── FIX 2: Ridge meta-learner instead of ElasticNetCV with Lasso option ──
# positive=True + l1_ratio=1.0 (Lasso) can zero-out MLP weights entirely
# Ridge keeps all models contributing; alpha is tuned via CV
from sklearn.linear_model import RidgeCV
meta = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], cv=5)
meta.fit(oof_tr, y_tr)

final_preds = meta.predict(pred_te)
r2_final    = r2_score(y_te, final_preds)
mae_final   = mean_absolute_error(y_te, final_preds)
rmse_final  = np.sqrt(mean_squared_error(y_te, final_preds))

print()
print('=' * 58)
print('  FINAL STACKED ENSEMBLE  [v5 fixed]')
print('=' * 58)
print(f'  R²   = {r2_final:.4f}')
print(f'  MAE  = {mae_final:.4f} log1p(barrier×10)')
print(f'  RMSE = {rmse_final:.4f}')
print()
print('  Individual model R²:')
for j, name in enumerate(ALL_MODEL_NAMES):
    w = meta.coef_[j] if hasattr(meta, 'coef_') else '—'
    print(f'  {name:6s}: R²={r2_score(y_te, pred_te[:, j]):.4f}  weight={w:.4f}')
print()
if r2_final >= 0.90:
    print('  🎉 TARGET R² ≥ 0.90 ACHIEVED!')
elif r2_final >= 0.85:
    print('  ✅ ≥0.85 — run Cell 9 (Optuna) for final push')
else:
    print('  Run Cell 9 (Optuna) — expect +0.03–0.05 more')


In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 8 — Diagnostic plots
# ════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Convert back to eV space for intuitive residuals
barrier_pred_ev  = (np.expm1(final_preds)) / 10
barrier_true_ev  = (np.expm1(y_te.values)) / 10

ax = axes[0]
ax.scatter(barrier_true_ev, barrier_pred_ev, alpha=0.3, s=8, color='royalblue')
ax.plot([0, 2], [0, 2], 'r--', lw=2)
ax.set_xlabel('True barrier (eV)'); ax.set_ylabel('Predicted barrier (eV)')
ax.set_title(f'Barrier space  R²={r2_final:.4f}')
ax.set_xlim(0, 2); ax.set_ylim(0, 2)

resid = y_te.values - final_preds
ax = axes[1]
ax.scatter(final_preds, resid, alpha=0.3, s=8, color='darkorange')
ax.axhline(0, color='red', lw=2, ls='--')
ax.set_xlabel('Predicted log1p'); ax.set_ylabel('Residual')
ax.set_title('Residual plot')

ax = axes[2]
ax.hist(resid, bins=60, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(0, color='red', lw=2)
ax.set_xlabel('Residual'); ax.set_title(f'Residual dist  MAE={mae_final:.3f}')

plt.suptitle(f'SSE Surrogate v5  —  R² = {r2_final:.4f}', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(str(OUT_DIR / 'v5_results.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Plot saved')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 9 — Optuna LGB tuning  [v5 — bugs fixed]
#
# Fixes vs original v5:
#   1. Param keys now FULL names (not short aliases that corrupt on replace)
#   2. best_lgb built from best_p (full keys), not study.best_params (short)
#   3. OOF pred_new accumulates fold-by-fold (not direct single predict)
#      so it matches the distribution of all other models in pred_te
# ════════════════════════════════════════════════════════════
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from sklearn.model_selection import cross_val_score

def objective(trial):
    # FIX: use full parameter names — no renaming needed, no corruption risk
    p = {
        'n_estimators':      trial.suggest_int('n_estimators',      600, 3000),
        'learning_rate':     trial.suggest_float('learning_rate',   0.003, 0.04, log=True),
        'max_depth':         trial.suggest_int('max_depth',         5, 12),
        'num_leaves':        trial.suggest_int('num_leaves',        30, 200),
        'min_child_samples': trial.suggest_int('min_child_samples', 3, 30),
        'subsample':         trial.suggest_float('subsample',       0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree',0.4, 1.0),
        'reg_lambda':        trial.suggest_float('reg_lambda',      0.01, 10.0, log=True),
        'reg_alpha':         trial.suggest_float('reg_alpha',       0.0, 1.0),
        'n_jobs': -1, 'verbose': -1, 'random_state': 42,
    }
    return cross_val_score(
        lgb.LGBMRegressor(**p), X_tr, y_tr,
        cv=5, scoring='r2', n_jobs=1
    ).mean()

print('Running Optuna (60 trials, full param names)...')
study = optuna.create_study(direction='maximize',
                             sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=60, show_progress_bar=True)

print(f'\nBest CV R²: {study.best_value:.4f}')
print('Best params:', study.best_params)

# FIX: best_lgb uses study.best_params directly (already full names now)
best_lgb = lgb.LGBMRegressor(**study.best_params, n_jobs=-1, verbose=-1, random_state=42)
best_lgb.fit(X_tr, y_tr)
r2_opt = r2_score(y_te, best_lgb.predict(X_te))
print(f'Tuned LGB alone → R² = {r2_opt:.4f}')

j_lgb = ALL_MODEL_NAMES.index('lgb')
if r2_opt > r2_score(y_te, pred_te[:, j_lgb]):
    print('Replacing LGB in ensemble with fold-averaged OOF...')
    oof_new  = oof_tr.copy()
    pred_new = pred_te.copy()
    # FIX: accumulate fold-by-fold — matches how all other pred_te columns were built
    pred_new[:, j_lgb] = 0.0
    for fold, (ti, vi) in enumerate(kf.split(X_tr, y_tr)):
        best_lgb.fit(X_tr.iloc[ti], y_tr.iloc[ti])
        oof_new[vi, j_lgb]   = best_lgb.predict(X_tr.iloc[vi])
        pred_new[:, j_lgb]  += best_lgb.predict(X_te) / kf.n_splits

    # Refit meta on updated OOF
    from sklearn.linear_model import RidgeCV
    meta_new = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], cv=5)
    meta_new.fit(oof_new, y_tr)
    new_preds = meta_new.predict(pred_new)
    r2_new    = r2_score(y_te, new_preds)
    print(f'Updated ensemble → R² = {r2_new:.4f}')
    if r2_new > r2_final:
        final_preds = new_preds
        r2_final    = r2_new
        meta        = meta_new
        oof_tr      = oof_new
        pred_te     = pred_new
        print(f'🎉 New best: R² = {r2_final:.4f}')
        if r2_final >= 0.90:
            print('🎉 TARGET R² ≥ 0.90 ACHIEVED!')


In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 10 — Predict full dataset + SSE candidates  [v5 fixed]
#
# Fix: uses qt_inference (fitted on full X_tr in Cell 7)
#      and MLP_MODELS (refitted on full X_tr in Cell 7)
# ════════════════════════════════════════════════════════════
kB = 8.617333e-5  # eV/K

full_base  = np.zeros((len(X), n_total))
X_full_qt  = qt_inference.transform(X)   # FIX: qt_inference not qt

for j, (name, model) in enumerate(BASE_MODELS.items()):
    full_base[:, j] = model.predict(X)

for k, (name, model) in enumerate(MLP_MODELS.items()):
    j = len(BASE_MODELS) + k
    full_base[:, j] = model.predict(X_full_qt)  # FIX: fully retrained MLP

log_pred_full = meta.predict(full_base)
barrier_pred  = np.clip((np.expm1(log_pred_full)) / 10, 0.005, 5.0)
log10_sigma   = -barrier_pred / (kB * 300) / np.log(10)

res = pd.DataFrame({
    'predicted_barrier_eV':  barrier_pred,
    'predicted_log10_sigma': log10_sigma,
}, index=X.index)

mask = res['predicted_barrier_eV'] < 0.4
if 'energy_above_hull' in X.columns: mask &= X['energy_above_hull'] < 0.05
if 'band_gap'          in X.columns: mask &= X['band_gap']          > 2.0

top = res[mask].sort_values('predicted_barrier_eV').head(20)
print(f'SSE candidates: {mask.sum()}')
print('\nTop 20:')
print(top.to_string())


In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 11 — Save
# ════════════════════════════════════════════════════════════
joblib.dump(BASE_MODELS,  str(OUT_DIR / 'v5_base_models.pkl'))
joblib.dump(MLP_MODELS,   str(OUT_DIR / 'v5_mlp_models.pkl'))
joblib.dump(meta,         str(OUT_DIR / 'v5_meta_model.pkl'))
joblib.dump(qt,           str(OUT_DIR / 'v5_quantile_transformer.pkl'))
joblib.dump(le_dict,      str(OUT_DIR / 'v5_label_encoders.pkl'))
res.to_csv(str(OUT_DIR   / 'v5_all_predictions.csv'))
top.to_csv(str(OUT_DIR   / 'v5_top_sse_candidates.csv'))

print('✅ Saved')
print()
print('=' * 58)
print('  v4 → v5 comparison')
print('=' * 58)
print(f'  v4 R² = 0.8128  (5 tree models, log(x+0.001) target)')
print(f'  v5 R² = {r2_final:.4f}  (+ CatBoost + MLP, log1p(x×10) target)')
print(f'  Δ R²  = {r2_final - 0.8128:+.4f}')
print('=' * 58)
if r2_final >= 0.90:
    print('  🎉 TARGET ACHIEVED!')